# Practice Problem: Cover Letter Assistant with Review and Rewind
Scenario

You’re building a LangGraph workflow that writes a cover letter for a job posting. A human checks the work at two points. Afterwards, the human can go back to any earlier point to try a different decision, without starting over.

In [ ]:
from typing import TypedDict,Literal,Annotated
from langgraph.graph import StateGraph,START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
import operator

class JobCover(TypedDict):
    job_description : str
    resume_text : str
    required_skills : list[str]
    resume_points : list[str]
    matched_points : list[str]
    draft : str
    feedback : Annotated[list[str],operator.add]
    revision_count : int
    final_letter : str
    status : str
    

# define nodes

parse_job: Takes a job description string and extracts 4–6 required skills. Use an LLM, or simple keyword matching if you want to save API calls.<br>
match_resume: Compares the skills against a hardcoded resume (a Python dict of 6–8 experience points from your own background) and picks the 3–4 most relevant points.<br>
confirm_points (HITL #1): Shows the matched points to the human. The human can:
approve them as-is, or
send back a list of indexes to remove.<br>
write_letter: Writes a short cover letter (about 150 words) from the confirmed points. If feedback exists in state, it uses that feedback in the rewrite.<br>
human_review (HITL #2): Shows the draft. The human can:
approve, which goes to finalize;
edit, which sends back corrected text and then goes to finalize; or
reject with feedback, which goes back to write_letter.<br>
finalize: Stores the final letter and sets status = "done".

In [ ]:
def parse_job(state : JobCover):
    print(f"node parse_job started*****")
    # for given job description:
    # required_skills = ["Python", "LangChain", "LangGraph", "RAG", "Vector databases",
    #                "Test automation", "Kubernetes", "AWS", "MLflow"]
    required_skills = ["Python", "Java", "Test automation", "LangChain", "Prompt engineering"]
    return {
        "required_skills" : required_skills
    }

def parse_resume (state : JobCover):
    print(f"node parse_resume started*****")
    resume_points = [
    "Built a Java test automation framework with Selenium and TestNG, 1,200+ regression tests",
    "Cut release regression time from 3 days to 6 hours with parallel test suites in Jenkins",
    "Developed REST APIs in Java and Spring Boot for a payments platform",
    "Built a PDF Q&A chatbot: RAG pipeline in Python with LangChain and FAISS vector databases",
    "Built a research assistant agent in LangGraph with human approval steps",
    "Ran prompt engineering experiments for test case generation from requirements",
    "Created a Docker setup to run the chatbot locally",
    "Mentored 4 junior QA engineers on automation design patterns",
    ]
    return {
        "resume_points" : resume_points
    }

def match_resume(state : JobCover):
    print(f"node match_resume started*****")
    matched_points = ["Built a PDF Q&A chatbot: RAG pipeline in Python with LangChain and FAISS vector databases",
    "Built a research assistant agent in LangGraph with human approval steps",
    "Ran prompt engineering experiments for test case generation from requirements",
    "Created a Docker setup to run the chatbot locally",
    "Mentored 4 junior QA engineers on automation design patterns",]
    return {
        "matched_points" :matched_points
    }

def confirm_details(state : JobCover)->Command[Literal["write_letter"]]:
    print(f"node confirm_details started*****")
    decision = interrupt({"value ":state["matched_points"],"question":"Approve or send back list of idices to remove"})

    remove = set(decision.get("indexList", []))
    kept = [p for i, p in enumerate(state["matched_points"]) if i not in remove]
    
    if len(decision["indexList"]) > 0:
        return Command(goto = "write_letter",update ={"matched_points":kept})
    return Command(goto = "write_letter")
    
    
def write_letter(state : JobCover):
    print(f"node write_letter started*****")
    draft = "Written draft for resume and db using feed back if available, its less than 150 words. Its just a toy data."
    return {
        "draft" : draft
    }
    

def human_review(state : JobCover)->Command[Literal["finalize","write_letter","rejectNodeDropped"]]:
    print(f"node human_review started*****")
    decision = interrupt({"question":"approve, edit with corrected text or reject with feedback","payload" : state["draft"]})
    print(f"********{decision}")
    if decision["act"] == "approve":
        print(f"********approve")
        return Command(goto="finalize")
    elif decision["act"] == "edit":
        print(f"********edit")
        return Command(goto="finalize", update = {"draft":decision["editedDraft"]})
    else:
        print(f"********rejected")
        revision_cnt = state.get("revision_count",0) +1
        print(f"********{revision_cnt}")
        if revision_cnt > 3:
            print(f"********{revision_cnt}")
            return Command(goto="rejectNodeDropped")
        return Command(goto="write_letter",update={"feedback":[decision["feedback"]],"revision_count":revision_cnt})
    
def rejectNodeDropped(state : JobCover)->Command[Literal["finalize","rejectNodeDropped"]]:
    print(f"node rejectNodeDropped started*****")
    appEdit = interrupt({"question":"approve, edit with corrected text.Reject is invalid since you rejected 3 times.","payload" : state["draft"]})
    if appEdit["act"] == "approve":
        return Command(goto="finalize")
    elif appEdit["act"] == "edit":
        return Command(goto="finalize", update = {"draft":appEdit["editedDraft"]})
    return Command(goto="rejectNodeDropped")   # invalid input: ask again


def finalize(state:JobCover):
     print(f"node finalize started*****")
     return{
         "final_letter":state["draft"],
         "status":"Done"
     }


node human_review started*****
node finalize started*****
node write_letter started*****
node human_review started*****

In [ ]:
build = StateGraph(state_schema=JobCover)

build.add_node("parse_job",parse_job)
build.add_node("parse_resume",parse_resume)
build.add_node("match_resume",match_resume)
build.add_node("confirm_details",confirm_details)
build.add_node("write_letter",write_letter)
build.add_node("human_review",human_review)
build.add_node("finalize",finalize)
build.add_node("rejectNodeDropped",rejectNodeDropped)

# add edges
build.add_edge(START, "parse_job")
build.add_edge(START, "parse_resume")
build.add_edge("parse_job", "match_resume")
build.add_edge("parse_resume", "match_resume")
build.add_edge("match_resume", "confirm_details")
# build.add_edge("confirm_details", "write_letter")
build.add_edge("write_letter", "human_review")

build.add_edge("finalize", END)

memory = InMemorySaver()

workflow = build.compile(checkpointer=memory)

# from IPython.display import Image
# Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
jd_with_gaps = """
Company: Nimbus Analytics (Bengaluru, Hybrid)
Role: LLM Engineer, 3–6 years experience

About the role:
We are building an AI assistant that helps enterprise customers search and
summarize internal documents. You will design retrieval pipelines, build
agent workflows, and take them to production.

Responsibilities:
- Build RAG pipelines over PDFs and internal wikis using LangChain and LangGraph
- Design multi-step agent workflows with human approval steps
- Write automated tests and evaluation suites for LLM outputs
- Deploy and scale services on Kubernetes in AWS
- Track experiments and model versions with MLflow
- Work with product and QA teams to define quality metrics

Required skills:
- Python
- LangChain / LangGraph
- RAG and vector databases
- Test automation
- Kubernetes
- AWS
- MLflow

Nice to have:
- Java or another backend language
- Experience with prompt evaluation frameworks
"""

resume_text = """
Priya Raman
AI/ML Engineer | Bengaluru

SUMMARY
Software engineer with 7 years of experience in Java development and test
automation, now working on LLM applications with Python and LangChain.

EXPERIENCE
Senior SDET, Orbit Fintech (2021 - 2025)
- Built a test automation framework in Java with Selenium and TestNG, covering 1,200+ regression tests
- Reduced release regression time from 3 days to 6 hours by parallelizing test suites in Jenkins
- Mentored 4 junior QA engineers on automation design patterns

Java Developer, Crestline Systems (2018 - 2021)
- Developed REST APIs in Java and Spring Boot for a payments platform
- Wrote unit and integration tests with JUnit and Mockito

PROJECTS
- PDF Q&A chatbot: RAG pipeline in Python using LangChain, FAISS vector databases, and OpenAI embeddings
- Research assistant agent: multi-step workflow in LangGraph with human approval steps
- Prompt engineering experiments: compared zero-shot and few-shot prompts for test case generation from requirements
- Built a Docker setup to run the chatbot locally

SKILLS
Python, Java, Spring Boot, LangChain, LangGraph, RAG, Vector databases,
Prompt engineering, Test automation, Selenium, Docker, Git, Jenkins
"""

In [ ]:
config = {"configurable":{"thread_id":"user123"}}

In [ ]:
# workflow.invoke(input={"job_description":jd_with_gaps,"resume_points":resume_text},config=config)

In [ ]:
# workflow.get_state(config=config).next

In [ ]:
# workflow.invoke(Command(resume ={"indexList":[0,2]}),config=config)

In [ ]:
jd_no_gaps = """
Company: Ledgerline Software (Bengaluru, Onsite)
Role: AI Engineer, Quality Automation, 4–8 years experience

About the role:
Our QA platform team is adding LLM features to our test automation tools.
You will build features that generate and review test cases using LLMs.

Responsibilities:
- Build LLM-powered tools that generate test cases from requirements
- Integrate LangChain-based pipelines into our existing Java test framework
- Write automated tests for both traditional code and LLM features
- Review and improve prompts based on QA team feedback
"""

In [ ]:
for chunk in workflow.stream(input={"job_description":jd_with_gaps,"resume_text":resume_text},stream_mode="updates",config=config):
    print(chunk)

In [ ]:
test = workflow.invoke(input={"job_description":jd_with_gaps,"resume_text":resume_text},config=config)

In [ ]:
#Print the interrupt payload each time the graph pauses.

# test["__interrupt__"][0].value

# 1st interupt

In [ ]:
# test1= workflow.invoke(Command(resume ={"indexList":[0,2]}),config=config)
# test1["__interrupt__"][0].value

# 2nd interupt

In [ ]:
# test1= workflow.invoke(Command(resume ={"act":"approve"}),config=config)
# test1= workflow.invoke(Command(resume ={"act":"edit","editedDraft":"Edited ,Written draft for resume and db using feed back [] if available, its less than 150 words. Its just a toy data."}),config=config)
# test1 = workflow.invoke(Command(resume ={"act":"reject","feedback":"feedback1"}),config=config)
# test1 = workflow.invoke(Command(resume ={"act":"reject","feedback":"feedback1"}),config=config)
# test1 = workflow.invoke(Command(resume ={"act":"reject","feedback":"feedback1"}),config=config)
# test1 = workflow.invoke(Command(resume ={"act":"reject","feedback":"feedback1"}),config=config)
# test1["__interrupt__"][0].value
# test1= workflow.invoke(Command(resume ={"act":"approve"}),config=config)
# test1

Part A: HITL
Run one full session where you:
Remove one matched point at confirm_points.
Reject the first draft with the feedback "make it more concise".
Approve the second draft.


In [ ]:
workflow.invoke(Command(resume ={"indexList":[0,2]}),config=config)
workflow.invoke(Command(resume ={"act":"reject","feedback":"make it more concise"}),config=config)


In [ ]:
final_result = workflow.invoke(Command(resume ={"act":"approve"}),config=config)

In [ ]:
for s in workflow.get_state_history(config = config):
    print(s.next)

In [ ]:
workflow.get_state(config = config).values['final_letter']

# Part B: Time travel
After Part A finishes, do the following on the same thread:<br>
1. B1. History dump <br>
 Print the full checkpoint history, one line per checkpoint: step number, next node(s), revision_count, and the first 40 characters of draft.<br>
2. B2. Replay<br>
 Find the checkpoint just before the first write_letter ran. Re-run from it without changing anything. Before running, write down your predictions:<br>
Which nodes will print started?<br>
Will the LLM be called again?<br>
Will the graph pause at human_review again?<br>
3. B3. Fork by changing state<br>
 Go back to the checkpoint after match_resume, before confirm_points. Change state so 
that matched_points has only 2 items, then continue the run to the end. Confirm that the original timeline still appears in the history.<br>
4. B4. Fork at an interrupt<br>
 Go back to the checkpoint where human_review was paused on draft 1, and this time resume with approve. You should now have a finished letter in which revision_count is 0.


In [ ]:
# B1

for s in workflow.get_state_history(config = config):
    print(f"step number :{s.metadata['step']}, next node :{s.next},revision_cnt :{s.values.get('revision_count')},draft : {s.values.get('draft','')[:40]}")

In [ ]:
for s in workflow.get_state_history(config = config):
    print(s.next,s.config["configurable"])



In [ ]:
# B2

# Which nodes will print started? - node write_letter started*****"
# Will the LLM be called again? - yes
# Will the graph pause at human_review again? - yes

prevSnap = [s for s in workflow.get_state_history(config = config) if s.next == ('write_letter',) and s.values.get('revision_count',0)==0]
travelConfig = prevSnap[0].config
print(travelConfig)
workflow.invoke(None,config = travelConfig)
workflow.invoke(Command(resume ={"act":"approve"}),config=config)



In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

B3. Fork by changing state
Go back to the checkpoint after match_resume, before confirm_points. Change state so that matched_points has only 2 items, then continue the run to the end. Confirm that the original timeline still appears in the history.

In [ ]:
for s in workflow.get_state_history(config = config):
    print(s.next,s.config["configurable"])

In [ ]:
forkPoint = [s for s in workflow.get_state_history(config = config) if s.next == ('confirm_details',)][0]

forkPoint


In [ ]:
newFork = workflow.update_state(config=forkPoint.config,values = {"matched_points" : ["Built a PDF Q&A chatbot: RAG pipeline in Python with LangChain and FAISS vector databases",
    "Built a research assistant agent in LangGraph with human approval steps"]})

workflow.invoke(None,config=newFork)





In [ ]:
invoke1 = workflow.invoke(Command(resume ={"indexList":[]}),config=config)
invoke1

In [ ]:
workflow.invoke(Command(resume ={"act":"approve"}),config=config)

In [ ]:
for s in workflow.get_state_history(config=config):
    print(s.metadata["source"], s.values.get("matched_points")) # origin

B4. Fork at an interrupt
 Go back to the checkpoint where human_review was paused on draft 1, and this time resume with approve. You should now have a finished letter in which revision_count is 0.

In [ ]:
for s in workflow.get_state_history(config=config):
    print(s.next,s.values.get("revision_count"),s.config["configurable"]["checkpoint_id"] ,s.interrupts)

In [ ]:
interruptFork = [ s for s in workflow.get_state_history(config=config) if s.next == ('human_review',) and s.values.get("revision_count",0)==0]
newConfig = interruptFork[-1].config

In [ ]:
b4fork = workflow.update_state(config = newConfig,values={})
workflow.invoke(None,config = b4fork)

In [ ]:
# b4 result checked , revision count should be zero
workflow.invoke(Command(resume = {"act":"approve"}),config=config).get("revision_count",0)

In [ ]:
for s in workflow.get_state_history(config=config):
    print(s.next,s.values.get("revision_count"),s.config["configurable"]["checkpoint_id"] ,s.interrupts)

In [ ]:
workflow.get_state(config=config)

# Questions to answer in comments
1. In B2, did the replay cost you LLM calls? Why or why not? - its toy data, my assumption : yes, if LLM is there, rerun will be done from write_letter , because checkpoint has next node as write_letter<br>
2. In B3, which node did LangGraph treat as the author of your state change, and why does that matter for which node runs next? - author node will be  confirm_details, because node itself is confirm_details. <br>
3. After B4, how many "final" letters exist across all timelines in this thread? Which one does graph.get_state(config) return? - including HITL and other invokes with approval - 4 final_letters, get_state returns last execution snapshot<br>
4. How is a fork different from just starting a new thread_id with the same input?- fork comes out of existing thread_id, new thread_id is completely different graph<br>


Your answers <br>

1.  Correct reasoning. Nodes before the checkpoint aren’t re-run; everything after it is re-executed, so write_letter runs again. Your B2 prediction still lists only write_letter, though. human_review also prints started, because the print runs before interrupt() pauses it.<br>

2.  Not correct. The author was match_resume, not confirm_details. At your fork checkpoint, next was ('confirm_details',), which means confirm_details hadn’t run yet. When you don’t pass as_node, LangGraph treats your update as if it came from the node that last wrote state, which was match_resume. That’s why confirm_details ran next and asked for approval again.<br>

Try it with as_node="confirm_details" and predict first what runs next. Remember that confirm_details now routes only through Command, with no static edge.<br>

3.  Correct. 4 final letters (Part A, B2, B3, B4), and get_state returns the most recently created checkpoint, which is B4’s.<br>

4.  Correct, but add the key benefit. A fork reuses everything before the fork point: earlier nodes don’t re-run, LLM calls aren’t repeated, and earlier human answers aren’t asked again. Both timelines also stay in one history, so you can compare them. A new thread_id starts from scratch and repeats all of that.<br>